# agents and tools implementation

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool
from langchain import hub
from langchain_community.tools import TavilySearchResults
from datetime import datetime

from dotenv import load_dotenv
load_dotenv()

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

In [2]:
GOOGLE_MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.5

llm = ChatGoogleGenerativeAI(model=GOOGLE_MODEL, temperature=TEMPERATURE)
search_tool = TavilySearchResults(max_results=5, search_depth='basic')

/var/folders/r8/0pmp7dnn1xb83tyx9ls8nt2c0000gp/T/ipykernel_7356/3952332407.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=5, search_depth='basic')


In [3]:
@tool
def get_system_time(format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """Get the current system time in the specified format."""
    return datetime.now().strftime(format)

In [5]:
tools = [search_tool, get_system_time]

prompt = hub.pull('hwchase17/react')

agent = create_react_agent(tools=tools, llm=llm, prompt=prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [9]:
# user_query = "write me a funny tweet about the current whether in Mumbai"
user_query = "When was SpaceX's last launch and how many days ago was that from this instant"

response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: SpaceX last launch date[{'title': 'SpaceX FINALLY Rolled out Starship Flight 10 Booster to Pad...But ...', 'url': 'https://www.youtube.com/watch?v=M8JrVsRL6Aw', 'content': 'SpaceX FINALLY Rolled out Starship Flight 10 Booster to Pad...But Launch Date Delays?\nGREAT SPACEX\n183000 subscribers\n1617 likes\n28293 views\n21 Aug 2025\nSpaceX FINALLY Rolled out Starship Flight 10 Booster to Pad...But Launch Date Delays?\n===\n00:00: Intro\n00:35: Flight 10 schedule delay possibility\n01:22: B16’s movement and more\n05:44: Pad-2’s progress\n07:37: NASA’s new operational direction\n===\n #greatspacex #elonmusk #spacex #nasa #starship \n==\nAdvertisers who want to place ads on our channel, please contact the email manager: smanager339@gmail.com\n===\nSpaceX Starship SN\nBe the first to sponsor us Thank you.\nhttps://www.patreon.com/GreatspaceX?fan_landing=true\nOur video content is referenced by video sourc

In [10]:
print(response)

{'input': "When was SpaceX's last launch and how many days ago was that from this instant", 'output': "SpaceX's last launch was on August 20, 2025. This was 3 days ago from the current instant (August 23, 2025)."}
